# 10 — Advanced

Scheduler, fallback LLM chain, background audio, noise filter, custom STT/TTS, custom LLM HTTP.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


### Local mode
Construct a Patter instance with a Twilio carrier.


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises scheduler, IVR/DTMF, background audio, and the EventBus.


### Scheduler


In [ ]:
import { scheduleOnce, scheduleInterval, scheduleCron } from "getpatter";
await cell('scheduler', { tier: 1, env }, () => {
  const h1 = scheduleOnce(new Date(Date.now() + 3_600_000), () => {});
  const h2 = scheduleInterval(300, () => {});
  const h3 = scheduleCron('0 9 * * 1-5', () => {});
  console.log(`once:     ${h1.jobId.slice(0, 20)}...`);
  console.log(`interval: ${h2.jobId.slice(0, 20)}...`);
  console.log(`cron:     ${h3.jobId.slice(0, 20)}...`);
  h1.cancel(); h2.cancel(); h3.cancel();
  console.log('All handles cancelled');
});


### IVR / DTMF


In [ ]:
import { DtmfEvent, formatDtmf } from "getpatter";
await cell('ivr_dtmf', { tier: 1, env }, () => {
  const seq = [DtmfEvent.ONE, DtmfEvent.EIGHT, DtmfEvent.ZERO, DtmfEvent.ZERO];
  console.log(`digits:   ${seq.map(e => e.value || e)}`);
  console.log(`formatted: ${formatDtmf(seq)}`);
});


### Background audio


In [ ]:
import { BackgroundAudioPlayer, BuiltinAudioClip } from "getpatter";
await cell('background_audio', { tier: 1, env }, () => {
  console.log('Built-in clips:', Object.values(BuiltinAudioClip));
  const player = new BackgroundAudioPlayer(BuiltinAudioClip.HOLD_MUSIC, { volume: 0.15, loop: true });
  console.log(`Player: volume=${player.volume}  loop=${player.loop}`);
});


### EventBus


In [ ]:
import { EventBus } from "getpatter";
await cell('event_bus', { tier: 1, env }, () => {
  const bus = new EventBus();
  const received: any[] = [];
  const unsub = bus.on('call_ended', (p: any) => received.push(p));
  bus.emit('call_ended', { callId: 'c001', duration: 45 });
  bus.emit('call_ended', { callId: 'c002', duration: 120 });
  console.log(`Received ${received.length} events`);
  unsub();
  bus.emit('call_ended', { callId: 'c003' });
  console.log(`After unsub: ${received.length} (c003 not received)`);
  if (received.length !== 2) throw new Error('expected 2 events');
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.
